# Parte 3 — Engine de recomendações

Recebe uma campanha (uma linha do dataset) e devolve recomendações de melhoria quantificadas, contextuais e priorizadas. O código está em `src/engine.py`, e as decisões no [README](../README.md).

Roteiro: 1. Como funciona · 2. Exemplos · 3. Validação no teste

In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.engine import EngineRecomendacoes
from src.modelo import carregar_treino_teste

X_train, y_train, X_test, y_test = carregar_treino_teste()
engine = EngineRecomendacoes().fit(X_train, y_train)

## 1. Como funciona

Três tipos de recomendação:
- Criativo: muda um atributo do vídeo (hook, rosto, legenda, CTA, densidade de texto) e mede o ganho no score com o modelo pré-veiculação, com intervalo de 95% por bootstrap. O texto acrescenta o efeito observado no CTR e no engajamento na plataforma da campanha (ou na faixa etária, para rosto), só quando a diferença é significativa após a correção de Holm, o mesmo critério da EDA.
- Performance: para métricas abaixo da referência da plataforma (mediana, quartil superior ou 10% melhores), estima com o modelo pós-veiculação o ganho no score de chegar a essa referência.
- Mídia: retargeting, para objetivos de resposta (conversão, tráfego, instalação). Melhora conversão e ROAS, mas não o score.

Prioridade: maior ganho no score primeiro; mídia por último, porque não muda o score. Campanhas que já têm quase tudo recebem também o que manter, com quanto cada atributo sustenta do score. Os ganhos são estimados um a um e não devem ser somados.

## 2. Exemplos

In [2]:
def mostrar(campaign_id):
    campanha = pd.concat([X_test, y_test], axis=1).set_index("campaign_id", drop=False).loc[campaign_id]
    print(f"{campaign_id} | {campanha['platform']} | {campanha['category']} | objetivo: {campanha['objective']} | "
          f"público: {campanha['target_audience_age']} | klike_score real: {campanha['klike_score']}")
    print(f"hook: {campanha['has_hook']} | rosto: {campanha['has_face']} | CTA: {campanha['has_cta']} | "
          f"legenda: {campanha['has_subtitle']} | texto: {campanha['text_density']}\n")
    for _, linha in engine.recomendar(campanha).iterrows():
        print(f"{linha['prioridade']}. [{linha['tipo']}] {linha['recomendacao']}\n")


teste = pd.concat([X_test, y_test], axis=1)
exemplos = {
    "TikTok com o menor score": teste[teste["platform"] == "TikTok"].nsmallest(1, "klike_score")["campaign_id"].iloc[0],
    "LinkedIn com score mediano": teste[teste["platform"] == "LinkedIn"].sort_values("klike_score")["campaign_id"].iloc[8],
    "maior score do teste": teste.nlargest(1, "klike_score")["campaign_id"].iloc[0],
}
exemplos

{'TikTok com o menor score': 'KLK-0164',
 'LinkedIn com score mediano': 'KLK-0472',
 'maior score do teste': 'KLK-0052'}

In [3]:
mostrar(exemplos["TikTok com o menor score"])

KLK-0164 | TikTok | E-commerce | objetivo: traffic | público: 18-24 | klike_score real: 30.2
hook: False | rosto: False | CTA: False | legenda: False | texto: medium

1. [criativo] Adicionar um hook nos 3 primeiros segundos: pode aumentar o klike_score em cerca de 17,1 pontos (previsão de 41 para 58; IC 95%: +15,5 a +18,8); no TikTok, campanhas assim têm CTR mediano 83% maior (o desta passaria de 2,2% para cerca de 4,0%); nas três plataformas juntas, a taxa de engajamento mediana é 56% maior.

2. [criativo] Incluir um rosto humano no vídeo: pode aumentar o klike_score em cerca de 11,2 pontos (previsão de 41 para 52; IC 95%: +9,1 a +12,6); nas três plataformas juntas, a taxa de engajamento mediana é 51% maior.

3. [criativo] Adicionar legendas: pode aumentar o klike_score em cerca de 7,7 pontos (previsão de 41 para 48; IC 95%: +6,2 a +9,4).

4. [criativo] Incluir um call-to-action: pode aumentar o klike_score em cerca de 7,2 pontos (previsão de 41 para 48; IC 95%: +5,4 a +8,8).

5. [per

In [4]:
mostrar(exemplos["LinkedIn com score mediano"])

KLK-0472 | LinkedIn | Lead Gen | objetivo: engagement | público: 35-44 | klike_score real: 60.2
hook: True | rosto: True | CTA: False | legenda: False | texto: low

1. [criativo] Adicionar legendas: pode aumentar o klike_score em cerca de 7,7 pontos (previsão de 55 para 63; IC 95%: +6,2 a +9,4).

2. [criativo] Aumentar a quantidade de texto na tela de baixo para médio: pode aumentar o klike_score em cerca de 7,6 pontos (previsão de 55 para 63; IC 95%: +5,8 a +9,6).

3. [criativo] Incluir um call-to-action: pode aumentar o klike_score em cerca de 7,2 pontos (previsão de 55 para 62; IC 95%: +5,4 a +8,8).

4. [performance] O CTR desta campanha (8,0%) está abaixo do quartil superior das campanhas do LinkedIn (11,3%). Sugestão: testar em A/B variações da abertura e da primeira cena do vídeo. Chegar a esse nível elevaria o klike_score em cerca de 2,8 pontos.

5. [performance] A taxa de engajamento desta campanha (1,81%) está abaixo do quartil superior das campanhas do LinkedIn (2,76%). Suges

In [5]:
mostrar(exemplos["maior score do teste"])

KLK-0052 | TikTok | Lead Gen | objetivo: engagement | público: 18-24 | klike_score real: 92.4
hook: True | rosto: True | CTA: True | legenda: False | texto: medium

1. [criativo] Adicionar legendas: pode aumentar o klike_score em cerca de 7,7 pontos (previsão de 77 para 84; IC 95%: +6,2 a +9,4).

2. [performance] A taxa de engajamento desta campanha (3,67%) está abaixo do quartil superior das campanhas do TikTok (5,32%). Sugestão: incluir um convite explícito à interação (comentar, compartilhar). Chegar a esse nível elevaria o klike_score em cerca de 1,0 pontos.

3. [performance] A retenção (fração do vídeo assistida) desta campanha (45%) está abaixo da mediana das campanhas do TikTok (49%). Sugestão: revisar o ritmo do vídeo para segurar a audiência até o fim (ex: cortar trechos lentos). Chegar a esse nível elevaria o klike_score em cerca de 0,3 pontos.



## 3. Validação no teste

### 3.1 Cobertura

Todas as campanhas do teste precisam receber pelo menos 3 recomendações.

In [6]:
recomendacoes = {cid: engine.recomendar(linha, maximo=None) for cid, linha in X_test.set_index("campaign_id", drop=False).iterrows()}
quantidade = pd.Series({cid: len(r) for cid, r in recomendacoes.items()})
print(f"recomendações por campanha: mínimo {quantidade.min()}, mediana {quantidade.median():.0f}, máximo {quantidade.max()}")
pd.concat(recomendacoes.values())["tipo"].value_counts().rename("recomendações no teste").to_frame()

recomendações por campanha: mínimo 3, mediana 6, máximo 11


,recomendações no teste
tipo,
performance,288
criativo,220
mídia,55
manter,2


### 3.2 Os ganhos estimados se confirmam em dados novos?

O ganho de cada ação de criativo foi estimado no treino. Comparo com a diferença observada no teste, que o modelo nunca viu: a diferença entre as medianas de score de quem tem e de quem não tem o atributo.

In [7]:
limpo_teste = engine.modelo_pre_[0].transform(X_test).assign(klike_score=y_test)
coef = pd.Series(engine.modelo_pre_[-1].coef_, index=engine.modelo_pre_[1].get_feature_names_out())
comparacoes = {
    "hook": ("has_hook", 1.0, 0.0, coef["has_hook"]),
    "rosto humano": ("has_face", 1.0, 0.0, coef["has_face"]),
    "legenda": ("has_subtitle", 1.0, 0.0, coef["has_subtitle"]),
    "call-to-action": ("has_cta", 1.0, 0.0, coef["has_cta"]),
    "texto médio (vs alto)": ("text_density", "medium", "high", coef["text_density_medium"] - coef["text_density_high"]),
}
pd.DataFrame([
    {"ação": acao, "ganho estimado (treino)": estimado,
     "diferença observada (teste)": limpo_teste.loc[limpo_teste[col] == com, "klike_score"].median()
                                    - limpo_teste.loc[limpo_teste[col] == sem, "klike_score"].median(),
     "campanhas com / sem": f"{(limpo_teste[col] == com).sum()} / {(limpo_teste[col] == sem).sum()}"}
    for acao, (col, com, sem, estimado) in comparacoes.items()
]).set_index("ação").round(1)

,ganho estimado (treino),diferença observada (teste),campanhas com / sem
ação,,,
hook,17.1,16.2,50 / 50
rosto humano,11.2,14.0,56 / 44
legenda,7.7,4.5,56 / 34
call-to-action,7.2,11.5,68 / 32
texto médio (vs alto),15.1,5.9,50 / 22


Todas as ações têm o mesmo sinal no teste. Hook e rosto se confirmam bem (16,2 e 14,0 pontos observados, contra 17,1 e 11,2 estimados). As demais variam mais, como esperado com grupos de 22 a 68 campanhas e diferenças brutas, sem ajuste pelos outros atributos: legenda e texto médio aparecem menores no teste, e CTA maior. O efeito menos estável é o do texto médio, com só 22 campanhas de texto alto no teste.

### 3.3 Potencial de melhoria

Ganho no score da recomendação mais importante de cada campanha do teste:

In [8]:
principal = pd.Series({cid: r["ganho_score"].iloc[0] for cid, r in recomendacoes.items()})
principal.describe()[["mean", "25%", "50%", "75%", "max"]].round(1).rename("ganho da 1ª recomendação (pontos)").to_frame()

,ganho da 1ª recomendação (pontos)
mean,13.3
25%,10.3
50%,16.1
75%,17.1
max,17.1


Para metade das campanhas do teste, a primeira recomendação vale 16 pontos ou mais (em geral, adicionar um hook); para um quarto, 10 pontos ou menos.

## Limitações

- Os efeitos vêm de dados observacionais: campanhas com hook têm score maior, mas o dataset não prova que adicionar um hook causa o aumento. A forma de confirmar é um teste A/B.
- O modelo assume efeitos aditivos e iguais entre plataformas, o que os dados sustentam (passo 7 da EDA e checagem de interações no modelo), mas com só 66 campanhas de LinkedIn.
- O CTR projetado nunca passa de 15%, o teto observado nos dados (passo 4.4).
- As recomendações de performance dizem quanto vale melhorar uma métrica, mas a ação sugerida para isso (ex: testar outra abertura) é uma hipótese de marketing, não um resultado dos dados.